In [1]:
import pandas as pd
import boto3
import sagemaker
from sagemaker.sklearn.processing import SKLearnProcessor
region = 'us-west-2'
role = 'arn:aws:iam::454531037350:role/service-role/AmazonSageMaker-ExecutionRole-20211103T133222'


sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /Users/dileep/Library/Application Support/sagemaker/config.yaml


In [2]:
session = boto3.session.Session(profile_name = "dev_dataapps_admin-454531037350")

In [3]:

s3 = session.resource('s3')


In [4]:

my_bucket = s3.Bucket('data-apps-sample-datasets')

for my_bucket_object in my_bucket.objects.all():
    print(my_bucket_object.key)

lead_scoring/
lead_scoring/data_for_prediction.csv
lead_scoring/golden_dataset.csv
lead_scoring/lead_scoring_dataset.csv


In [5]:
from sagemaker.session import Session

In [6]:
sagemaker_session = Session(boto_session=session)

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /Users/dileep/Library/Application Support/sagemaker/config.yaml


In [17]:
import sys
print(sys.version)

3.8.17 (default, Jul  5 2023, 15:35:58) 
[Clang 14.0.6 ]


In [18]:
sklearn_processor = SKLearnProcessor(framework_version='1.2-1',
                                     role=role,
                                     #instance_type='ml.t3.medium',
                                     instance_type='local', 
                                     instance_count=1, 
                                     sagemaker_session=sagemaker_session,
                                     #sagemaker_session=sagemaker_session_local,
                                     base_job_name='test-job')

INFO:sagemaker.image_uris:Defaulting to only available Python version: py3


In [8]:
from sagemaker.processing import ProcessingInput, ProcessingOutput

In [9]:
import zipfile
import os
from fnmatch import fnmatch
import tempfile

TEMP_DIR = tempfile.gettempdir()

def can_exclude(name: str, exclude_list: list) -> bool:
    """Check if the name can be excluded

    Args:
        name: Name to exclude
        exlude_list: List of names to be excluded can be regular expressions

    Returns:
        bool: True if the name can be excluded
    """
    for exclude_name in exclude_list:
        if fnmatch(name, exclude_name):
            return True
    return False

def zip_directory(
    dir_path: str, exclude_folders: list = [], exclude_files: list = []
) -> str:
    """Zip the directory

    Args:
        dir_path (str):  Path to the directory to zip
        exclude_folders (list): List of directories to be excluded from the zip
        exclude_files (list): List of files to be excluded from the zip

    Returns:
        str: Path to the zip file

    Raises:
        ValueError: If the directory does not exist
    """
    if dir_path == '.':
        dir_path = os.getcwd()
    if not os.path.exists(dir_path):
        raise ValueError(f"Directory {dir_path} does not exist")

    dir_name = os.path.basename(dir_path)
    zip_path = os.path.join(TEMP_DIR, dir_name + ".zip")

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zip_file:
        for root, dirs, files in os.walk(dir_path):

            # Exclude folders
            dirs_to_remove = []
            for dirname in dirs:
                if can_exclude(dirname, exclude_folders):
                    dirs_to_remove.append(dirname)

            for dirname in dirs_to_remove:
                dirs.remove(dirname)

            for file in files:
                # Exclude files
                if can_exclude(file, exclude_files):
                    continue

                abs_file_path = os.path.join(root, file)
                zip_file.write(
                    abs_file_path, abs_file_path.replace(dir_path, "")
                )

    return zip_path




In [19]:
exclude_folders = ["__pycache__",  ".*", "logs", "output", "python_packages", "sample_profiles_project", 
                   "tests"]
exclude_files = [".*", "*.md", "*.log", "pb_script.py", "*.ipynb", "*.json", "*.yml"]
zip_path = zip_directory(os.getcwd(), exclude_folders, exclude_files)

In [11]:
import yaml
homedir = os.path.expanduser("~") 
with open(os.path.join(homedir, ".pb/siteconfig.yaml"), "r") as f:
    creds = yaml.safe_load(f)["connections"]["shopify_wh"]["outputs"]["dev"]

In [12]:
import json

In [13]:
sagemaker_code_path = '/opt/ml/processing/input'
sagemaker_output_path = "/opt/ml/processing/output"
processing_input = ProcessingInput(source=zip_path, destination=sagemaker_code_path)
script_params = {}
script_params["--input_path"]= sagemaker_code_path
script_params["--credentials"]= json.dumps(creds)
script_params["--output_path"]= sagemaker_output_path
script_params["--source_code_zip"] = os.path.join(processing_input.destination, os.path.basename(processing_input.source))
arguments = []
for k, v in script_params.items():
    arguments.append(f"{k}")
    arguments.append(f"{v}")

In [ ]:
sklearn_processor.run(
    code="sagemaker_processing.py",
    inputs=[ProcessingInput(source=zip_path, destination=sagemaker_code_path)],
    outputs=[
        ProcessingOutput(output_name="data.csv", 
                         source=sagemaker_output_path, 
                         destination="s3://data-apps-utils-template-project/sample"),
    ],
    arguments=arguments
)
